# Does There Reply? - Ridge Binary Classifier

Train on `train.csv`, evaluate on the supplied `validation.csv`, and create predictions for `test.csv`. Change the hyperparameters and rerun from the configuration cell.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.pipeline import Pipeline

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'doesthereply'

# Hyperparameters.
RIDGE_ALPHA = 10.0
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
MIN_DF = 1
THRESHOLD = 0.5

train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
X_train = train_df['text'].fillna('').astype(str)
X_dev = dev_df['text'].fillna('').astype(str)
X_test = test_df['text'].fillna('').astype(str)
y_train = train_df['label'].astype(int)
y_dev = dev_df['label'].astype(int)
print(f'Train: {len(train_df)} | validation: {len(dev_df)} | test: {len(test_df)}')

In [ ]:
from sklearn.compose import ColumnTransformer

features = ColumnTransformer([
    ('word_features', TfidfVectorizer(ngram_range=WORD_NGRAM_RANGE, min_df=MIN_DF, sublinear_tf=True), 'text'),
    ('char_features', TfidfVectorizer(analyzer='char', ngram_range=CHAR_NGRAM_RANGE, min_df=MIN_DF, sublinear_tf=True), 'text'),
])
model = Pipeline([('features', features), ('regressor', Ridge(alpha=RIDGE_ALPHA))])
model.fit(pd.DataFrame({'text': X_train}), y_train)
dev_scores = np.clip(model.predict(pd.DataFrame({'text': X_dev})), 0, 1)
dev_predictions = (dev_scores >= THRESHOLD).astype(int)
print(f'Validation MAE: {mean_absolute_error(y_dev, dev_predictions):.4f}')
print(f'Validation macro-F1: {f1_score(y_dev, dev_predictions, average="macro", zero_division=0):.4f}')

In [ ]:
model.fit(pd.DataFrame({'text': pd.concat([X_train, X_dev], ignore_index=True)}), pd.concat([y_train, y_dev], ignore_index=True))
test_scores = np.clip(model.predict(pd.DataFrame({'text': X_test})), 0, 1)
test_predictions = (test_scores >= THRESHOLD).astype(int)
submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
submission_path = DATA_DIR / 'submission_ridge.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
if 'label' in test_df.columns:
    print(f'Test macro-F1: {f1_score(test_df["label"], test_predictions, average="macro", zero_division=0):.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')
submission.head()